<a href="https://colab.research.google.com/github/aadityane93/Toxic_Comments_Sentiment_Analysis/blob/aaditya_initial/1_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Toxic Comment Classification

## Kaggle DataSet


In [1]:
#https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge

## Imports

In [8]:
%pip install pandas
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import numpy as np
from itertools import combinations

## Load Data

In [11]:
test_labels = pd.read_csv("test_labels.csv")
test_labels.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,-1,-1,-1,-1,-1,-1
1,0000247867823ef7,-1,-1,-1,-1,-1,-1
2,00013b17ad220c46,-1,-1,-1,-1,-1,-1
3,00017563c3f7919a,-1,-1,-1,-1,-1,-1
4,00017695ad8997eb,-1,-1,-1,-1,-1,-1


In [12]:
test_df = pd.read_csv(
    "test.csv",
    engine="python",
    on_bad_lines="skip"
)

test_df = test_df.dropna(subset=["comment_text"])
test_df["comment_text"] = test_df["comment_text"].astype(str)

test_df.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \r\n\r\n The title is fine as i...
2,00013b17ad220c46,""" \r\n\r\n == Sources == \r\n\r\n * Zawe Ashto..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [13]:
train_df = pd.read_csv(
    "train.csv",
    engine="python",
    on_bad_lines="skip"
)

train_df = train_df.dropna(subset=["comment_text"])
train_df["comment_text"] = train_df["comment_text"].astype(str)

train_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\r\nWhy the edits made under my use...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\r\nMore\r\nI can't make any real suggestions...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [19]:
# Label columns
label_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Basic overview
print("Training data shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

print("\nMissing values:")
print(train_df.isna().sum())

print("\nDuplicate IDs:")
print(train_df["id"].duplicated().sum())


Training data shape: (159571, 10)

Columns:
['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'label_count', 'any_toxic']

Missing values:
id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
label_count      0
any_toxic        0
dtype: int64

Duplicate IDs:
0


In [16]:
# Label distribution
label_counts = train_df[label_cols].sum().sort_values(ascending=False)

label_distribution = pd.DataFrame({
    "positive_count": label_counts,
    "positive_percent": (label_counts / len(train_df) * 100).round(2),
    "negative_count": len(train_df) - label_counts
})

label_distribution

,positive_count,positive_percent,negative_count
toxic,15294,9.58,144277
obscene,8449,5.29,151122
insult,7877,4.94,151694
severe_toxic,1595,1.00,157976
identity_hate,1405,0.88,158166
threat,478,0.30,159093


## Number of clean comment and toxic comment in the data 
### Is this comment toxic at all, or is it clean? In the dataset, each comment has six possible labels:
- toxic
- severe_toxic
- obscene
- threat
- insult
- identity_hate

### A comment is treated as toxic if any one of those labels is 1.

### A comment is treated as clean if all six labels are 0.

In [17]:
# Clean vs toxic comments
train_df["label_count"] = train_df[label_cols].sum(axis=1)
train_df["any_toxic"] = (train_df["label_count"] > 0).astype(int)

binary_distribution = train_df["any_toxic"].map({
    0: "clean",
    1: "toxic"
}).value_counts()

binary_distribution_df = pd.DataFrame({
    "count": binary_distribution,
    "percent": (binary_distribution / len(train_df) * 100).round(2)
})

binary_distribution_df

,count,percent
any_toxic,,
clean,143346,89.83
toxic,16225,10.17


## Comment Length

This section checks how long the comments are. We count both characters and words.

In [22]:
train_df["char_count"] = train_df["comment_text"].str.len()
train_df["word_count"] = train_df["comment_text"].str.split().str.len()

length_summary = train_df[["char_count", "word_count"]].describe()

length_summary

,char_count,word_count
count,159571.000000,159571.000000
mean,396.593961,67.273527
std,594.387869,99.230702
min,6.000000,1.000000
25%,97.000000,17.000000
50%,207.000000,36.000000
75%,438.000000,75.000000
max,5000.000000,1411.000000


## Comment Length by Clean and Toxic Groups

Here, we compare the length of clean comments and toxic comments. Just checking to see whether toxic comments are usually shorter or longer than clean comments.

In [24]:
train_df["toxicity_group"] = train_df["any_toxic"].map({
    0: "clean",
    1: "toxic"
})

length_by_group = train_df.groupby("toxicity_group")[["char_count", "word_count"]].agg(
    ["mean", "median", "max"]
).round(2)

length_by_group

char_count              word_count             
                     mean median   max       mean median   max
toxicity_group                                                
clean              406.89  218.0  5000      68.92   38.0  1250
toxic              305.67  129.0  5000      52.72   23.0  1411

In [26]:
clean_count = (train_df["any_toxic"] == 0).sum()
toxic_count = (train_df["any_toxic"] == 1).sum()
multi_label_count = (train_df["label_count"] > 1).sum()

print(f"The training dataset contains {len(train_df):,} comments.")
print(f"There are {clean_count:,} clean comments.")
print(f"There are {toxic_count:,} toxic comments.")
print(f"{multi_label_count:,} comments have more than one toxicity label.")
print("The dataset is imbalanced, so accuracy alone is not enough for evaluation.")

The training dataset contains 159,571 comments.
There are 143,346 clean comments.
There are 16,225 toxic comments.
9,865 comments have more than one toxicity label.
The dataset is imbalanced, so accuracy alone is not enough for evaluation.


## Training Data Summary

The training data is highly imbalanced. Most comments are clean, and only a small percentage are toxic.

This means accuracy alone is not enough to evaluate the model. We should also use precision, recall, and F1 score.